# Phase 6: Model A — Pre-Match Predictor Training

**Goal:** Train and evaluate 3 models on the 42-feature dataset. Pick the best one.

**Steps:**
1. Prepare data — time-based split + leakage validation
2. Train 3 models: Logistic Regression, Random Forest, XGBoost
3. Evaluate: accuracy, confusion matrix, ROC-AUC
4. Feature importance
5. Compare and pick best model
6. Save to `models/`

**Output:** `models/pre_match_model.pkl` + `models/scaler.pkl`

## 1. Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, RocCurveDisplay
)
from sklearn.model_selection import cross_val_score
from xgboost                 import XGBClassifier

plt.style.use('dark_background')
os.makedirs('charts', exist_ok=True)

full = pd.read_csv('../data/processed/full_features.csv', parse_dates=['date'])
print(f'Dataset: {full.shape[0]} matches, {full.shape[1]} columns')
print(f'Class balance — team_a won: {full["team_a_won"].mean():.1%}')

## 2. Prepare Data — Time-Based Split

Train: 2008–2023 | Test: 2024–2025

Why NOT random split? Because in the real world you train on the past and predict the future.
A random split would let the model see 2024 matches during training — that is leakage.

In [ ]:
FEATURE_COLS = [
    'ta_overall_wr', 'tb_overall_wr',
    'ta_venue_wr', 'tb_venue_wr',
    'ta_h2h_wr',
    'ta_batfirst_wr', 'tb_batfirst_wr',
    'ta_last5_wr', 'tb_last5_wr',
    'ta_last5_margin', 'tb_last5_margin',
    'ta_streak', 'tb_streak',
    'toss_winner_is_ta', 'toss_decision_bat',
    'venue_batfirst_wr',
    'ta_home', 'tb_home',
    'ta_season_wr', 'tb_season_wr',
    'match_num_in_season', 'is_playoff',
    'is_day_night', 'venue_avg_first_inn_score',
    'ta_avg_batting_sr', 'ta_avg_batting_avg', 'ta_best_batting_avg',
    'ta_avg_bowling_econ', 'ta_avg_bowling_sr', 'ta_best_bowling_econ',
    'ta_total_caps', 'ta_favorable_matchups', 'ta_avg_matchup_sr',
    'tb_avg_batting_sr', 'tb_avg_batting_avg', 'tb_best_batting_avg',
    'tb_avg_bowling_econ', 'tb_avg_bowling_sr', 'tb_best_bowling_econ',
    'tb_total_caps', 'tb_favorable_matchups', 'tb_avg_matchup_sr',
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in full.columns]
TARGET = 'team_a_won'

train = full[full['season_year'] <= 2023]
test  = full[full['season_year'] >= 2024]

X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test,  y_test  = test[FEATURE_COLS],  test[TARGET]

print(f'Train: {len(train)} matches ({train["season_year"].min()}–{train["season_year"].max()})')
print(f'Test : {len(test)}  matches ({test["season_year"].min()}–{test["season_year"].max()})')
print(f'Features: {len(FEATURE_COLS)}')

In [ ]:
# Leakage validation — features at test boundary should only reflect pre-2024 history
print('First 5 test matches — verify win rates look like pre-2024 history:')
print(test.head(5)[['date','team_a','team_b','ta_overall_wr','tb_overall_wr']].to_string(index=False))

In [ ]:
# Scale features — StandardScaler: mean=0, std=1 per feature
# fit on train ONLY, then apply same transform to test
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print('Scaling done.')

## 3. Train 3 Models

In [ ]:
# Model 1: Logistic Regression (baseline)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)
lr_train_acc = accuracy_score(y_train, lr.predict(X_train_s))
lr_test_acc  = accuracy_score(y_test,  lr.predict(X_test_s))
lr_auc       = roc_auc_score(y_test,   lr.predict_proba(X_test_s)[:,1])
print(f'Logistic Regression — Train: {lr_train_acc:.1%} | Test: {lr_test_acc:.1%} | AUC: {lr_auc:.3f}')

In [ ]:
# Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_train_acc = accuracy_score(y_train, rf.predict(X_train))
rf_test_acc  = accuracy_score(y_test,  rf.predict(X_test))
rf_auc       = roc_auc_score(y_test,   rf.predict_proba(X_test)[:,1])
print(f'Random Forest       — Train: {rf_train_acc:.1%} | Test: {rf_test_acc:.1%} | AUC: {rf_auc:.3f}')

In [ ]:
# Model 3: XGBoost
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=42, eval_metric='logloss', verbosity=0
)
xgb.fit(X_train, y_train)
xgb_train_acc = accuracy_score(y_train, xgb.predict(X_train))
xgb_test_acc  = accuracy_score(y_test,  xgb.predict(X_test))
xgb_auc       = roc_auc_score(y_test,   xgb.predict_proba(X_test)[:,1])
print(f'XGBoost             — Train: {xgb_train_acc:.1%} | Test: {xgb_test_acc:.1%} | AUC: {xgb_auc:.3f}')

## 4. Compare Models

In [ ]:
results = pd.DataFrame({
    'Model'      : ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Train Acc'  : [lr_train_acc, rf_train_acc, xgb_train_acc],
    'Test Acc'   : [lr_test_acc,  rf_test_acc,  xgb_test_acc],
    'AUC'        : [lr_auc, rf_auc, xgb_auc],
    'Overfit Gap': [lr_train_acc-lr_test_acc, rf_train_acc-rf_test_acc, xgb_train_acc-xgb_test_acc]
}).sort_values('Test Acc', ascending=False)

print(results.to_string(index=False))
print()
print('=== WHAT THIS TELLS YOU ===')
print('RF and XGBoost hit ~100% train accuracy but collapse on test.')
print('This is OVERFITTING — they memorised training noise, not real patterns.')
print('Logistic Regression has a tiny gap (3%) — it generalises far better.')
print()
print('WHY? IPL has genuine randomness (pitch, weather, form on the day).')
print('Complex models learn this noise. For noisy sports data, simpler often wins.')
print('This is a real ML insight — not a failure.')

In [ ]:
# Try a regularised XGBoost to reduce overfitting
xgb_reg = XGBClassifier(
    n_estimators=200,
    max_depth=3,           # shallower = less overfitting
    learning_rate=0.05,    # slower learning = better generalisation
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=1.0,         # L1 regularisation
    reg_lambda=2.0,        # L2 regularisation
    min_child_weight=5,
    random_state=42, eval_metric='logloss', verbosity=0
)
xgb_reg.fit(X_train, y_train)
xgb_reg_train = accuracy_score(y_train, xgb_reg.predict(X_train))
xgb_reg_test  = accuracy_score(y_test,  xgb_reg.predict(X_test))
xgb_reg_auc   = roc_auc_score(y_test,   xgb_reg.predict_proba(X_test)[:,1])

print(f'XGBoost (regularised) — Train: {xgb_reg_train:.1%} | Test: {xgb_reg_test:.1%} | AUC: {xgb_reg_auc:.3f} | Gap: {xgb_reg_train-xgb_reg_test:.1%}')
print(f'Logistic Regression   — Train: {lr_train_acc:.1%} | Test: {lr_test_acc:.1%}  | AUC: {lr_auc:.3f} | Gap: {lr_train_acc-lr_test_acc:.1%}')
print()
print('Pick whichever has the highest TEST accuracy with the smallest gap.')

In [ ]:
# 5-fold cross-validation on training data
print('5-fold CV on training set:')
for name, model, X in [
    ('Logistic Regression', lr,  X_train_s),
    ('Random Forest',       rf,  X_train),
    ('XGBoost',             xgb, X_train)
]:
    cv = cross_val_score(model, X, y_train, cv=5, scoring='accuracy')
    print(f'  {name:<22}: {cv.mean():.1%} ± {cv.std():.1%}')

## 5. Detailed Evaluation

In [ ]:
# Evaluate best model in detail — Logistic Regression
best_model = lr
y_pred = best_model.predict(X_test_s)
y_prob = best_model.predict_proba(X_test_s)[:,1]

print('=== CLASSIFICATION REPORT (Logistic Regression) ===')
print(classification_report(y_test, y_pred, target_names=['Team B Won','Team A Won']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred B Won','Pred A Won'],
            yticklabels=['Actual B Won','Actual A Won'])
axes[0].set_title('Confusion Matrix (Logistic Regression)', fontweight='bold')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1], color='#00D4FF')
axes[1].plot([0,1],[0,1],'--',color='gray',alpha=0.5)
axes[1].set_title(f'ROC Curve — AUC: {lr_auc:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/06_model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: charts/06_model_evaluation.png')

## 6. Feature Importance

In [ ]:
# For Logistic Regression: coefficients show feature importance
# Positive coefficient = feature pushes toward team_a winning
# Negative coefficient = feature pushes toward team_b winning
coef_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'coefficient': lr.coef_[0]
}).reindex(pd.Series(lr.coef_[0]).abs().sort_values(ascending=False).index)
coef_df = coef_df.sort_values('coefficient', key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top15 = coef_df.head(15)
colors = ['#2ECC71' if c > 0 else '#E74C3C' for c in top15['coefficient']]
ax.barh(top15['feature'][::-1], top15['coefficient'][::-1], color=colors[::-1], alpha=0.85)
ax.axvline(0, color='white', linewidth=0.8)
ax.set_xlabel('Coefficient (positive = helps team_a, negative = helps team_b)')
ax.set_title('Logistic Regression — Top 15 Feature Coefficients', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('charts/06_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 most influential features:')
print(coef_df.head(10).to_string(index=False))

## 7. Save Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)

# Save Logistic Regression as the primary model
joblib.dump(lr,     '../models/pre_match_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')

with open('../models/feature_cols.json', 'w') as f:
    json.dump(FEATURE_COLS, f)

with open('../models/model_info.json', 'w') as f:
    json.dump({
        'model_type'      : 'LogisticRegression',
        'requires_scaling': True,
        'test_accuracy'   : round(lr_test_acc, 4),
        'auc'             : round(lr_auc, 4),
        'n_features'      : len(FEATURE_COLS)
    }, f, indent=2)

print(f'Saved: models/pre_match_model.pkl  (Logistic Regression)')
print(f'Saved: models/scaler.pkl')
print(f'Saved: models/feature_cols.json   ({len(FEATURE_COLS)} features)')
print(f'Saved: models/model_info.json')
print(f'\nFinal model — Test accuracy: {lr_test_acc:.1%} | AUC: {lr_auc:.3f}')

## Your Tasks (Before Phase 7)

```
□ Compare the 3 models — LR won. Does that surprise you? Why?

□ Look at the confusion matrix:
  - How many times did the model predict A wins but B actually won?
  - Is the model better at predicting wins or losses?

□ Look at feature coefficients:
  - Which feature has the biggest positive coefficient? What does that mean?
  - Where does toss_winner_is_ta rank? Top or bottom?

□ Try changing max_depth=3 to max_depth=10 in the regularised XGBoost.
  Does the overfit gap grow? Why?

□ Add to interview notebook:
  Q: 'Why did Logistic Regression beat XGBoost here?'
  A: [your answer — talk about overfitting and noise in sports data]

  Q: 'What is AUC-ROC and why is it better than accuracy?'
  A: [your answer]

  Q: 'How did you prevent overfitting?'
  A: [your answer]

  Q: 'Why time-based split instead of random?'
  A: [your answer]
```